# 06. Итог по флагам 15 / 180, правило end v2 (вся история, без окна)

> **Пересчёт.** Основа — `petri_states_eval_2013.ipynb / petri_states_eval_all.ipynb, шаг 3` из папки `петри`; код расчёта тот же, пути и флаги — из `common/settings.py`. Расчёт — в `03` и `04`; здесь только чтение результатов. Сравнения с вариантом 1 (15 / 365) выполняются, только если он посчитан. Текст с числами и выводами написан по **прежнему** расчёту: после пересчёта сверить (`common/compare_with_old.py`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "common"))
import settings as cfg

PART = "all"                         # "2013" — с окном, "all" — без окна

import json
import pandas as pd
from IPython.display import display
import petri_states as ps
import align_parallel as ap

CONFIG = "long180_v2"                    # при RUN_V1 = True можно поставить "long180_v1"
LONG_CUSTODY_DAYS = cfg.CONFIGS[CONFIG]["long"]
OUT = cfg.out_dir(PART, CONFIG)
OUT_365 = cfg.out_dir(PART, "long365_v1")
RESULTS = OUT / "split_eval.jsonl"


def done_models():
    return {r["model"]: r for r in ap.jl_read(RESULTS)}

## Итог

In [2]:
res = pd.DataFrame(done_models().values())
q = ps.summarize_split(res)
cols = ["model", "fitness_tbr_train", "fitness_tbr_test", "pct_fit_traces_tbr_test", "fitness_align_test",
        "pct_fit_traces_align_test", "align_timeout", "align_traces", "precision_etc_test", "precision_norm_test",
        "f1_tbr", "f1_align", "places", "transitions", "silent", "total_sec", "finished"]
q[[c for c in cols if c in q]].round(3)

,model,fitness_tbr_train,fitness_tbr_test,pct_fit_traces_tbr_test,fitness_align_test,pct_fit_traces_align_test,align_timeout,align_traces,precision_etc_test,precision_norm_test,f1_tbr,f1_align,places,transitions,silent,finished
0,flower,1.000,1.000,100.000,NaN,NaN,NaN,NaN,0.204,0.000,0.339,NaN,3,14,2,2026-11-05 21:06
1,HM_dep0.95,0.984,0.984,0.000,NaN,NaN,NaN,2295.0,0.358,0.193,0.525,NaN,16,43,31,2026-11-05 21:06
2,DFG_p0.05,0.957,0.958,50.065,0.921,49.020,0.0,2295.0,0.620,0.522,0.753,0.741,14,34,1,2026-11-05 21:27
3,DFG_p0.0,1.000,1.000,100.000,1.000,100.000,0.0,2295.0,0.262,0.072,0.415,0.415,14,83,1,2026-11-05 21:27
4,SplitMiner,0.761,0.761,7.407,0.217,7.407,0.0,2295.0,0.811,0.762,0.785,0.342,21,35,23,2026-11-05 21:27
5,IM_noise0.2,0.984,0.984,33.115,0.939,33.246,9.0,2295.0,0.453,0.313,0.621,0.612,30,41,31,2026-11-05 21:27


## После досчёта тайм-аутов (`04_alignments.ipynb`, шаг 2)

In [3]:
f = OUT / "split_eval_timeouts_fixed.csv"
display(pd.read_csv(f, encoding="utf-8-sig").round(4)) if f.exists() else print("нет", f)

,experiment,model,align_limit_s,traces,timeout_before,timeout_after,fitness_align_before,fitness_align_after,fitness_align_after_upper,pct_fit_traces_before,pct_fit_traces_after,precision_etc_test,precision_norm_test,f1_align_before,f1_align_after,fixed_how
0,all,DFG_p0.05,20,2295,0,0,0.9211,0.9211,0.9211,49.0196,49.0196,0.6196,0.5220,0.7409,0.7409,NaN
1,all,DFG_p0.0,20,2295,0,0,1.0000,1.0000,1.0000,100.0000,100.0000,0.2619,0.0724,0.4151,0.4151,NaN
2,all,SplitMiner,20,2295,0,0,0.2167,0.2167,0.2167,7.4074,7.4074,0.8105,0.7618,0.3420,0.3420,NaN
3,all,IM_noise0.2,20,2295,9,0,0.9391,0.9387,0.9387,33.2458,33.1155,0.4535,0.3132,0.6116,0.6115,align_600s: 9


## Сравнение с расчётом на всём журнале

In [4]:
# сравнение с расчётом без разбиения (01_states_all_miners.ipynb, весь журнал)
f = OUT_365 / "model_quality.csv"     # весь журнал, флаги 15 / 365 (01_states_all_miners.ipynb)
if f.exists():
    full = pd.read_csv(f).set_index("model")[["fitness_tbr", "precision_etc", "precision_norm", "f1"]]
    full.columns = [c + " | весь журнал" for c in full.columns]
    split = q.set_index("model")[["fitness_tbr_test", "fitness_align_test", "precision_etc_test", "precision_norm_test", "f1_tbr", "f1_align"]]
    split.columns = [c + " | test" for c in split.columns]
    display(split.join(full, how="left").round(3))
else:
    print("нет", f, "— сначала выполнить 01_states_all_miners.ipynb")

,fitness_tbr_test | test,fitness_align_test | test,precision_etc_test | test,precision_norm_test | test,f1_tbr | test,f1_align | test,fitness_tbr | весь журнал,precision_etc | весь журнал,precision_norm | весь журнал,f1 | весь журнал
model,,,,,,,,,,
flower,1.000,NaN,0.204,0.000,0.339,NaN,1.000,0.226,0.000,0.369
HM_dep0.95,0.984,NaN,0.358,0.193,0.525,NaN,0.984,0.417,0.246,0.585
DFG_p0.05,0.958,0.921,0.620,0.522,0.753,0.741,0.957,0.702,0.615,0.810
DFG_p0.0,1.000,1.000,0.262,0.072,0.415,0.415,1.000,0.302,0.098,0.464
SplitMiner,0.761,0.217,0.811,0.762,0.785,0.342,0.726,0.887,0.854,0.798
IM_noise0.2,0.984,0.939,0.453,0.313,0.621,0.612,0.984,0.524,0.385,0.684


### Сравнение с прежними флагами (SHORT = 15, LONG = 365)

In [5]:
f = OUT_365 / "split_eval.jsonl"
if f.exists():
    old = ps.summarize_split(pd.DataFrame([json.loads(l) for l in f.read_text(encoding="utf-8").splitlines() if l.strip()]))
    keep = ["fitness_tbr_test", "fitness_align_test", "align_timeout", "precision_etc_test", "precision_norm_test", "f1_tbr", "f1_align"]
    cmp = (q.set_index("model")[[c for c in keep if c in q]].add_suffix(f" | LONG={LONG_CUSTODY_DAYS}")
           .join(old.set_index("model")[[c for c in keep if c in old]].add_suffix(" | LONG=365")))
    cmp = cmp[sorted(cmp.columns, key=lambda c: (keep.index(c.split(" | ")[0]), c))]
    display(cmp.round(3))
else:
    print("нет", f)

,fitness_tbr_test | LONG=180,fitness_tbr_test | LONG=365,fitness_align_test | LONG=180,fitness_align_test | LONG=365,align_timeout | LONG=180,align_timeout | LONG=365,precision_etc_test | LONG=180,precision_etc_test | LONG=365,precision_norm_test | LONG=180,precision_norm_test | LONG=365,f1_tbr | LONG=180,f1_tbr | LONG=365,f1_align | LONG=180,f1_align | LONG=365
model,,,,,,,,,,,,,,
flower,1.000,1.000,NaN,NaN,NaN,NaN,0.204,0.189,0.000,0.000,0.339,0.317,NaN,NaN
HM_dep0.95,0.984,0.984,NaN,NaN,NaN,NaN,0.358,0.358,0.193,0.209,0.525,0.525,NaN,NaN
DFG_p0.05,0.958,0.957,0.921,0.919,0.0,0.0,0.620,0.620,0.522,0.531,0.753,0.752,0.741,0.740
DFG_p0.0,1.000,NaN,1.000,NaN,0.0,NaN,0.262,NaN,0.072,NaN,0.415,NaN,0.415,NaN
SplitMiner,0.761,0.720,0.217,0.114,0.0,0.0,0.811,0.884,0.762,0.857,0.785,0.793,0.342,0.202
IM_noise0.2,0.984,0.984,0.939,0.937,9.0,9.0,0.453,0.453,0.313,0.326,0.621,0.621,0.612,0.611


## ✅ Напоминание: проверить результаты

1. **Все ли модели посчитаны** — в итоговой таблице 5 строк (4 модели + `flower`), у всех заполнено `finished`.
2. **Тайм-ауты alignments**: доля `align_timeout / align_traces`. Если больше ~10% — `fitness_align_test` ненадёжна (в неё входят только «быстрые» трассы); указать это в тексте или пересчитать с большим лимитом / на `N_CORES > 1`.
3. **Переобучение**: `fitness_tbr_train` и `fitness_tbr_test` должны быть близки (разница > 0.02 — сеть подогнана под train).
4. **Сравнение с расчётом на всём журнале** (последняя таблица): precision на test обычно ниже — это нормально; если порядок моделей по F1 поменялся, отметить.
5. **TBR и alignments**: если `fitness_align_test` сильно ниже `fitness_tbr_test` (как у Split Miner), TBR завышает fitness из-за тихих переходов — в выводах опираться на alignments.
6. Записать итоговые числа и выбор лучшей модели в отчёт / `Требования_к_архитектуре.md`.